[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1RuwxBMQiWajzAo2wvU-L_prc9Hw_UZip/view?usp=drive_link)

# LLM Evaluation – Full Dataset

This notebook demonstrates how to evaluate LLM responses when you already have question–answer pairs. Each sample includes `user_input` and `llm_response`; Floeval runs metrics directly without generating responses.

**Objectives**
- Install Floeval and configure credentials
- Load a full dataset from in-memory samples
- Configure the LLM provider and evaluation metrics
- Run the evaluation and inspect aggregate and per-sample results

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
%pip install git+https://github.com/FloTorch/floeval.git@dev

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass
# LLM and API configuration

OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("your-api-key")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 3. Imports

The following cell imports the evaluation components and the LLM configuration schema. RAGAS metrics require both a chat model and an embedding model for scoring.

In [ ]:
from floeval import Evaluation, DatasetLoader
from floeval.config.schemas.io.llm import OpenAIProviderConfig

## 2. Configure the LLM

The LLM configuration is built using `OpenAIProviderConfig`. Set `OPENAI_API_KEY` in your environment or replace the placeholder. RAGAS metrics use both the chat model and the embedding model for answer relevancy scoring.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)

## 5. Load the Dataset

A full dataset is created from in-memory samples using `DatasetLoader.from_samples`. Each sample must include `user_input` and `llm_response`. The `partial_dataset=False` flag indicates that responses are already present.

In [ ]:
dataset = DatasetLoader.from_samples(
    [
        {"user_input": "What is Python?", "llm_response": "Python is a programming language."},
        {"user_input": "What is RAG?", "llm_response": "RAG stands for Retrieval-Augmented Generation."},
    ],
    partial_dataset=False,
)
print(f"Dataset loaded: {len(dataset.samples)} samples")

## 6. Create and Run the Evaluation

The `Evaluation` class is instantiated with the dataset, LLM config, and metrics. The `answer_relevancy` metric from the RAGAS provider is used to score how relevant each answer is to the question. The `run()` method executes the evaluation.

In [ ]:
evaluation = Evaluation(
    dataset=dataset,
    llm_config=llm_config,
    metrics=["answer_relevancy"],
    default_provider="ragas",
    metric_params={"answer_relevancy": {"threshold": 0.8}},
)

results = evaluation.run()
print("Aggregate scores:", results.aggregate_scores)


## 5. Inspect Per-Sample Results

Each sample in `results.sample_results` contains a `metrics` dictionary with the score, pass/fail status, and metadata for each metric. This enables per-sample analysis of evaluation quality.

In [ ]:
for i, sr in enumerate(results.sample_results, start=1):
    print(f"Sample {i}: {sr['user_input'][:50]}...")
    for key, data in sr.get("metrics", {}).items():
        print(f"  {key}: score={data.get('score')}, passed={data.get('passed')}")

## Summary

This notebook demonstrated the end-to-end process of evaluating LLM responses using a full dataset with Floeval.

The key components included:

1. **Dataset Loading**: A full dataset with `user_input` and `llm_response` for each sample was loaded using `DatasetLoader.from_samples`.
2. **LLM Configuration**: The OpenAI-compatible provider was configured with chat and embedding models for RAGAS metrics.
3. **Evaluation Execution**: The `answer_relevancy` metric was run via the RAGAS provider to score answer quality.
4. **Results Inspection**: Aggregate scores and per-sample metrics were accessed through `results.aggregate_scores` and `results.sample_results`.

This example showcases the standard workflow for evaluating pre-generated LLM outputs with Floeval.